<a href="https://colab.research.google.com/github/jnahMoch/revisedBookRecommender/blob/main/Books_categorizing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

booksDF = pd.read_csv('cleaned_books_dataset.csv')
booksDF['cleaned_description'] = booksDF['cleaned_description'].fillna('')

booksDF.head(10)

,isbn13,isbn10,title,authors,categories,description,thumbnail,average_rating,num_pages,cleaned_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,A NOVEL THAT READERS and critics have been eag...,http://books.google.com/books/content?id=KQZCP...,3.85,247.0,novel reader critic eagerly anticipating decad...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Fiction,A new 'Christie for Christmas' -- a full-lengt...,http://books.google.com/books/content?id=gA5GP...,3.83,241.0,new christie christmas fulllength novel adapte...
2,9780006163831,0006163831,The One Tree,Stephen R. Donaldson,Fiction,Volume Two of Stephen Donaldson's acclaimed se...,http://books.google.com/books/content?id=OmQaw...,3.97,479.0,volume two stephen donaldsons acclaimed second...
3,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,"A memorable, mesmerizing heroine Jennifer -- b...",http://books.google.com/books/content?id=FKo2T...,3.93,512.0,memorable mesmerizing heroine jennifer brillia...
4,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Religion,Lewis' work on the nature of love divides love...,http://books.google.com/books/content?id=XhQ5X...,4.15,170.0,lewis work nature love divide love four catego...
5,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Religion,"""In The Problem of Pain, C.S. Lewis, one of th...",http://books.google.com/books/content?id=Kk-uV...,4.09,176.0,problem pain c lewis one renowned christian au...
6,9780006380832,0006380832,Empires of the Monsoon,Richard Hall,History,Until Vasco da Gama discovered the sea-route t...,http://books.google.com/books/content?id=MuPEQ...,4.41,608.0,vasco da gama discovered searoute east almost ...
7,9780006470229,000647022X,The Gap Into Madness,Stephen R. Donaldson,Fiction,A new-cover reissue of the fourth book in the ...,http://books.google.com/books/content?id=4oXav...,4.15,743.0,newcover reissue fourth book bestselling fivev...
8,9780006472612,0006472613,Master of the Game,Sidney Sheldon,Fiction,Kate Blackwell is an enigma and one of the mos...,http://books.google.com/books/content?id=TkTYp...,4.11,489.0,kate blackwell enigma one powerful woman world...
9,9780006479673,0006479677,If Tomorrow Comes,Sidney Sheldon,Fiction,One of Sidney Sheldon's most popular and bests...,http://books.google.com/books/content?id=l2tBi...,4.04,501.0,one sidney sheldons popular bestselling title ...


In [3]:
booksDF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3759 entries, 0 to 3758
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   isbn13               3759 non-null   int64  
 1   isbn10               3759 non-null   object 
 2   title                3759 non-null   object 
 3   authors              3759 non-null   object 
 4   categories           3759 non-null   object 
 5   description          3759 non-null   object 
 6   thumbnail            3759 non-null   object 
 7   average_rating       3759 non-null   float64
 8   num_pages            3759 non-null   float64
 9   cleaned_description  3759 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 293.8+ KB


In [4]:
booksDF.size

37590

In [5]:
binary_mapping = {
    'Fiction': 'fiction',
    'Juvenile Fiction': 'fiction',
    'Comics & Graphic Novels': 'fiction',
    'Drama': 'fiction',
    'Poetry': 'fiction',
    'Biography & Autobiography': 'nonfiction',
    'History': 'nonfiction',
    'Literary Criticism': 'nonfiction',
    'Philosophy': 'nonfiction',
    'Religion': 'nonfiction',
    'Juvenile Nonfiction': 'nonfiction'
}
booksDF['binary_category'] = booksDF['categories'].map(binary_mapping)
train_df = booksDF.dropna(subset=['binary_category']).copy()

X = train_df['cleaned_description']
y = train_df['binary_category']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [7]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [8]:
model = LinearSVC(class_weight='balanced', C=0.5, random_state=42)
model.fit(X_train_vec, y_train)

LinearSVC(C=0.5, class_weight='balanced', random_state=42)

In [9]:
y_pred = model.predict(X_test_vec)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Accuracy: 0.8617


In [10]:
booksDF['simple_categories_binary'] = model.predict(vectorizer.transform(booksDF['cleaned_description']))

In [11]:
booksDF.drop(columns=['binary_category'], errors='ignore', inplace=True)
booksDF.to_csv('cleaned_books_dataset_simple_categories2.csv', index=False)

In [12]:
booksDF.head()

,isbn13,isbn10,title,authors,categories,description,thumbnail,average_rating,num_pages,cleaned_description,simple_categories_binary
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,A NOVEL THAT READERS and critics have been eag...,http://books.google.com/books/content?id=KQZCP...,3.85,247.0,novel reader critic eagerly anticipating decad...,fiction
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Fiction,A new 'Christie for Christmas' -- a full-lengt...,http://books.google.com/books/content?id=gA5GP...,3.83,241.0,new christie christmas fulllength novel adapte...,fiction
2,9780006163831,0006163831,The One Tree,Stephen R. Donaldson,Fiction,Volume Two of Stephen Donaldson's acclaimed se...,http://books.google.com/books/content?id=OmQaw...,3.97,479.0,volume two stephen donaldsons acclaimed second...,fiction
3,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,"A memorable, mesmerizing heroine Jennifer -- b...",http://books.google.com/books/content?id=FKo2T...,3.93,512.0,memorable mesmerizing heroine jennifer brillia...,fiction
4,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Religion,Lewis' work on the nature of love divides love...,http://books.google.com/books/content?id=XhQ5X...,4.15,170.0,lewis work nature love divide love four catego...,nonfiction
